# Module 01 Case Study: NYC 311 Service Requests

**Dataset:** NYC 311 Service Requests (synthetic 10 000-row sample mirroring the real schema)  
**Goal:** Answer four operational questions a city analyst might face on a Monday morning.

## The Questions We Will Answer

1. Which borough generates the most 311 complaints?
2. What are the top-10 complaint types city-wide?
3. Do complaint volumes vary by day of week?
4. How quickly does each borough close complaints on average?

---

## 0. Setup — imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

## 1. Generate the Synthetic Dataset

We simulate a realistic 311 dataset so this notebook runs offline without
downloading files. The distributions mirror the real NYC 311 open data.

In [ ]:
N = 10_000

boroughs = ['BROOKLYN', 'QUEENS', 'MANHATTAN', 'BRONX', 'STATEN ISLAND']
borough_weights = [0.30, 0.25, 0.22, 0.18, 0.05]

complaint_types = [
    'Noise - Residential', 'HEAT/HOT WATER', 'Illegal Parking',
    'Blocked Driveway', 'Street Light Condition', 'Noise - Street/Sidewalk',
    'UNSANITARY CONDITION', 'Request Large Bulky Item Collection',
    'Rodent', 'Graffiti', 'Derelict Vehicle', 'Water System',
    'PAINT/PLASTER', 'Plumbing', 'Noise - Commercial'
]
complaint_weights = [
    0.15, 0.12, 0.10, 0.09, 0.08, 0.07, 0.06,
    0.06, 0.05, 0.05, 0.05, 0.04, 0.04, 0.02, 0.02
]

statuses = ['Closed', 'Open', 'Pending']
status_weights = [0.80, 0.12, 0.08]

created_dates = pd.date_range('2023-01-01', periods=N, freq='1h') + \
    pd.to_timedelta(np.random.randint(0, 60, N), unit='min')

resolution_hours = np.abs(np.random.normal(72, 48, N)).clip(1, 720)
closed_dates = created_dates + pd.to_timedelta(resolution_hours, unit='h')

df = pd.DataFrame({
    'Unique Key': range(1, N + 1),
    'Created Date': created_dates,
    'Closed Date': np.where(
        np.random.choice([True, False], N, p=[0.80, 0.20]),
        closed_dates, pd.NaT
    ),
    'Complaint Type': np.random.choice(complaint_types, N, p=complaint_weights),
    'Borough': np.random.choice(boroughs, N, p=borough_weights),
    'Status': np.random.choice(statuses, N, p=status_weights),
})

# Introduce a small proportion of missing/messy borough values
bad_idx = np.random.choice(df.index, size=150, replace=False)
df.loc[bad_idx[:75], 'Borough'] = 'Unspecified'
df.loc[bad_idx[75:], 'Borough'] = np.nan

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Initial Inspection

Before writing a single line of analysis, always ask: **What do I actually have?**

In [ ]:
df.info()

In [ ]:
# Quick numeric summary
df.describe(datetime_is_numeric=True)

In [ ]:
# Count missing values per column
df.isnull().sum()

**Observations:**
- `Closed Date` has NaT entries (open complaints have no close date — expected).
- `Borough` has 75 NaN entries we need to handle before grouping.

## 3. Data Cleaning

### 3a. Normalise borough names

In [ ]:
print('Before cleaning:')
print(df['Borough'].value_counts(dropna=False))

In [ ]:
def clean_borough(val):
    if pd.isna(val) or str(val).strip().upper() in ('UNSPECIFIED', 'N/A', ''):
        return 'Unknown'
    return str(val).strip().title()

df['Borough'] = df['Borough'].apply(clean_borough)

print('After cleaning:')
print(df['Borough'].value_counts())

### 3b. Derive useful date features

In [ ]:
df['day_of_week'] = df['Created Date'].dt.day_name()
df['hour'] = df['Created Date'].dt.hour
df['month'] = df['Created Date'].dt.month

# Resolution time in hours (only where Closed Date exists)
df['resolution_hours'] = (
    df['Closed Date'] - df['Created Date']
).dt.total_seconds() / 3600

df[['Created Date', 'Closed Date', 'day_of_week', 'resolution_hours']].head()

## 4. Question 1 — Which borough generates the most complaints?

In [ ]:
borough_counts = (
    df[df['Borough'] != 'Unknown']
    .groupby('Borough')['Unique Key']
    .count()
    .sort_values(ascending=True)  # ascending for horizontal bar chart
)

fig, ax = plt.subplots()
bars = ax.barh(borough_counts.index, borough_counts.values, color='steelblue')
ax.bar_label(bars, padding=4, fmt='%,.0f')
ax.set_xlabel('Number of Complaints')
ax.set_title('311 Complaint Volume by Borough')
plt.tight_layout()
plt.show()

print(f'Top borough: {borough_counts.idxmax()} with {borough_counts.max():,} complaints')

## 5. Question 2 — Top-10 complaint types city-wide

In [ ]:
top10 = df['Complaint Type'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10.index[::-1], top10.values[::-1], color='teal')
ax.bar_label(ax.containers[0], padding=4, fmt='%,.0f')
ax.set_xlabel('Count')
ax.set_title('Top 10 Complaint Types — NYC 311')
plt.tight_layout()
plt.show()

top10

## 6. Question 3 — Do complaint volumes vary by day of week?

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

daily = (
    df.groupby('day_of_week')['Unique Key']
    .count()
    .reindex(day_order)
)

fig, ax = plt.subplots()
ax.plot(daily.index, daily.values, marker='o', color='coral', linewidth=2)
ax.fill_between(daily.index, daily.values, alpha=0.15, color='coral')
ax.set_ylabel('Complaint Count')
ax.set_title('311 Complaints by Day of Week')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print(f'Busiest day: {daily.idxmax()} ({daily.max():,} complaints)')
print(f'Quietest day: {daily.idxmin()} ({daily.min():,} complaints)')

## 7. Question 4 — Average resolution time by borough

In [ ]:
resolution = (
    df[df['Borough'] != 'Unknown']
    .dropna(subset=['resolution_hours'])
    .groupby('Borough')['resolution_hours']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'Mean (hrs)', 'median': 'Median (hrs)', 'count': 'N'})
    .round(1)
    .sort_values('Mean (hrs)')
)

print(resolution)

fig, ax = plt.subplots()
x = range(len(resolution))
ax.bar(x, resolution['Mean (hrs)'], label='Mean', alpha=0.7, color='steelblue')
ax.bar(x, resolution['Median (hrs)'], label='Median', alpha=0.7, color='coral', width=0.4)
ax.set_xticks(list(x))
ax.set_xticklabels(resolution.index, rotation=15)
ax.set_ylabel('Hours to Resolution')
ax.set_title('311 Resolution Time by Borough')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

| Question | Finding |
|----------|---------|
| Highest complaint borough | Brooklyn (~30% of all complaints) |
| #1 complaint type | Noise - Residential |
| Day-of-week pattern | Relatively flat — city never sleeps |
| Fastest resolution | Varies; median ≈ 60–80 hours across boroughs |

**Key Pandas patterns used in this notebook:**
- `groupby` + `count` / `agg` for aggregation
- `.dt` accessor for date feature engineering
- Boolean masks with `.loc` for filtering before grouping
- `value_counts()` for frequency tables
- `reindex` to impose a custom sort order on categorical data